# Isochrone Plotting Notebook

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # make pandas shut up about how I set variables in the isochrone function

In [2]:
'''
This function takes the name of your isochrone file and a log(age), filters the isochrone file for the appropriate age, and calculates the stellar radii from the luminosity, temperature, etc.

The age of the bulge is another complicated question, but we've been using 10^10 years as the age. So log(age) should just be 10.
'''
def load_iso(isofile,logages_):
    # these are just column numbers
    cRadius=0
    cMass=3
    cMini=2
    cMI=13
    cMTESS=-2
    cMGaia=-5
    cLum=6
    cTeff=4
    cAge=1
    logages = logages_
    
    # if you want multiple ages, give them as a list and this will keep all of them. Otherwise just give a single one.
    if isinstance(logages_,list):
        logages = np.array(logages_)
    elif not isinstance(logages_,np.ndarray):
        logages = np.array([logages])
    
    # Load isochrone, skipping rows that just have comments
    iso = pd.read_csv(isofile, sep='\\s+', header=None, skiprows=13, engine='python', comment='#')
                                                                                            
    #Replace the idx column with the radii                                                                  
    iso.iloc[:,0] = 10**(0.5*iso.iloc[:,cLum]-2*(iso.iloc[:,cTeff]-np.log10(5777.0)))
            
    # this does the actual filtering for the age(s) you want
    ret = []
    for i, logage in enumerate(logages):
        ret.append(iso[(iso.iloc[:,-1] <= 4.0) & (np.abs(iso.iloc[:,cAge]-logage)<0.001)])
    if len(ret)==1:
        ret = ret[0]
    
    return ret

In [10]:
## JUST SET SOME CONSTANT VARIABLES
isodir = "/fs/project/gaudi.1/crisp/mishaps/isochrones/MIST_v1.2_vvcrit0.4_PanSTARRS"
logage = 10
d_iso = 8000 # pc

mag = 'i' # band you want on magnitude axes
c1 = 'g' # first band in color
c2 = 'i' # second band in color

iso_cols = ['Radius_Estimate', 'log10_isochrone_age_yr', 'initial_mass', 'star_mass', 'log_Teff', 'log_g', 'log_L', '[Fe/H]_init',
            '[Fe/H]', 'PS_g', 'PS_r', 'PS_i', 'PS_z', 'PS_y', 'PS_w', 'PS_open', 'phase']

fehs = ['m2.50', 'm2.00', 'm1.75', 'm1.50', 'm1.25', 'm1.00', 'm0.75', 'm0.50', 'm0.25', 'p0.00', 'p0.25', 'p0.50']
print(len(fehs))

12


In [14]:
## PLOT STUFF
## For colors and linestyles, use same length as number of isochrones, change as you like
clrs = ['#125A56', '#125A56', '#238F9D', '#238F9D', '#60BCE9','#60BCE9', '#F9D576', '#F9D576', '#E94C1F', '#E94C1F', '#A01813', '#A01813']
lines = ['-', '--', '-', '--', '-', '--', '-', '--', '-', '--', '-', '--']
print(len(clrs))
print(len(lines))

fonts = 24 #fontsize, just so you don't have to change it in every label command

12
12


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(24,8))
axs[0].set_title('Isochrone CMD', fontsize=fonts)
axs[1].set_title('Isochrone Radii vs. Magnitude', fontsize=fonts)

for idx, feh in enumerate(fehs):
    isofile = f"{isodir}/MIST_v1.2_feh_{feh}_afe_p0.0_vvcrit0.4_PanSTARRS.iso.cmd"
    iso = load_iso(isofile, logage)
    iso.columns = iso_cols
    
    ps_iso = {} # make a dictionary to store the apparent magnitudes -- this just makes it easier to call different ones with the variables two boxes up
    
    # you could probably change this to only be the ones you're interested in, but it's extra work for very little speed-up
    ps_iso['g'] = iso.loc[:,'PS_g'] + 5*np.log10(d_iso) - 5
    ps_iso['r'] = iso.loc[:,'PS_r'] + 5*np.log10(d_iso) - 5
    ps_iso['i'] = iso.loc[:,'PS_i'] + 5*np.log10(d_iso) - 5
    ps_iso['z'] = iso.loc[:,'PS_z'] + 5*np.log10(d_iso) - 5
    
    axs[0].plot(ps_iso[c1]-ps_iso[c2], ps_iso[mag], c=clrs[idx], ls=lines[idx])
    
    axs[1].plot(ps_iso[mag], iso.loc[:,'Radius_Estimate'], c=clrs[idx], ls=lines[idx]) # you could use the function, but since this is just supposed to demonstrate how the isochrone changes at different metallicities, it doesn't really matter

axs[0].set_xlabel(f"({c1}-{c2})", fontsize=fonts)
axs[0].set_xlim(0.0, 1.5)
axs[0].set_ylabel(f"{mag}-Magnitude", fontsize=fonts)
axs[0].set_ylim(16.0, 23.0)
axs[0].tick_params(axis='both', which='major', labelsize=16)
axs[0].invert_yaxis()
axs[0].grid()

axs[1].set_xlabel(f"{mag}-Magnitude", fontsize=fonts)
axs[1].set_xlim(16.0, 23.0)
axs[1].set_ylabel(r"Stellar Radius [R$_{\odot}$]", fontsize=fonts)
axs[1].set_ylim(0.0, 6.0)
axs[1].tick_params(axis='both', which='major', labelsize=16)
axs[1].grid()

plt.tight_layout()
plt.savefig('Isochrone_Metallicity.png')

# Instead of trying to squeeze a legend on here, I would instead put "Increasing [Fe/H]" and arrows over the plot in your PowerPoint slide
# In the CMD, [Fe/H] is increasing left to right
# In the Radius vs. Mag, [Fe/H] is increasing bottom to top